# 프롬프트 버전과 A/B 비교

같은 과제를 세 가지 프롬프트 전략으로 실행하고 토큰·지연·단어 포함 표시를 한 표에 어떻게 기록할까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

`run(messages)`는 이 노트북에서 작성하는 보조 함수입니다. 역할·내용 목록을 받아 `model.invoke()`를 한 번 실행하고, 답 문자열·사용량·경과 초를 담은 딕셔너리를 반환합니다. `def` 셀은 함수를 정의할 뿐이며 뒤에서 `run(...)`을 호출할 때 모델 요청이 시작됩니다. `mention_flags(text)`는 지정 단어가 문자열에 있는지 표시한 딕셔너리를 반환합니다. 답의 뜻을 판단하는 모델 호출은 포함하지 않습니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## Ch 6 측정 기록 연결

Ch 6이 저장한 제공자 측정 기록을 읽고 공통 필드와 파일 지문을 확인합니다.

`run_id`는 실행을 구분하는 이름이고 SHA-256은 파일 내용을 바이트로 읽어 계산한 지문입니다. `read_text()`는 텍스트를, `read_bytes()`는 바이트를 읽습니다. `hexdigest()`는 계산된 지문을 16진수 문자열로 표현합니다. 이 값은 내용 변경을 비교하는 데 쓰며 사실 정확성을 평가하는 점수는 아닙니다.

`next((path for path in ... if ...), None)`는 조건에 맞는 첫 경로를 찾고 없으면 `None`을 반환합니다. `[Path.cwd(), *Path.cwd().parents]`에서 별표는 상위 경로들을 목록 안에 펼칩니다. 현재 폴더부터 상위 폴더를 찾는 준비 코드이며 모델 입력을 만드는 단계와 구분합니다.

`required <= ch06_record.keys()`는 필요한 키 집합이 저장 기록의 키에 모두 포함되는지 검사합니다. 이어지는 기록은 같은 필드 형식을 참고하지만 고객 문의와 Ch 6의 질문이 다르므로 두 실행 수치를 성능 변화로 비교하지 않습니다.

In [5]:
# 프롬프트 원문과 raw 응답을 UTF-8 JSON으로 저장한 뒤 다시 읽어 확인합니다.
from hashlib import sha256
from pathlib import Path

course_root = next((path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / "labs/part2/ch06-langchain-start").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("실습 묶음의 labs/part2 폴더를 찾지 못했습니다.")
ch06_relative = Path("labs/part2/ch06-langchain-start/runs/provider-comparison-continuity.json")
ch06_path = course_root / ch06_relative
if not ch06_path.exists():
    raise FileNotFoundError("Ch 6의 06 노트북을 먼저 실행하세요.")
ch06_record = json.loads(ch06_path.read_text(encoding="utf-8"))
ch06_sha256 = sha256(ch06_path.read_bytes()).hexdigest()
required = {"run_id", "run_date", "source_chapter", "conditions", "pricing", "records"}
assert required <= ch06_record.keys()
assert ch06_record["pricing"]["openai"] == {"ordinary_input": 0.75, "cached_input": 0.075, "output": 4.50}
print("Ch 6 기록:", ch06_record["run_id"], ch06_sha256[:12])

Ch 6 기록: ch06-provider-20260909T173655923454Z 531b30d0c1a0


## 버전과 변경 이유 기록

버전마다 바꾼 조건을 한 가지 전략으로 설명합니다.

In [6]:
# v3는 v2 제약을 유지한 채 관련 예시만 추가합니다.
versions = {
    "v1-direct": "답하세요. " + TASK,
    "v2-constraints": "분류/다음 행동/고객 문장 세 줄로 답하고 규정 밖 약속은 하지 마세요. " + TASK,
    "v3-example": "예시: 파손 상품은 교환으로 분류하고 사진과 주문번호를 확인합니다. "
                  "분류/다음 행동/고객 문장 세 줄로 답하고 규정 밖 약속은 하지 마세요. " + TASK,
}
change_log = {"v1-direct": "기준선", "v2-constraints": "형식·금지 조건",
              "v3-example": "v2 조건 유지·관련 예시 추가"}
print(json.dumps(change_log, ensure_ascii=False, indent=2))

{
  "v1-direct": "기준선",
  "v2-constraints": "형식·금지 조건",
  "v3-example": "v2 조건 유지·관련 예시 추가"
}


## 세 전략 실행

모델, 질문, 생성 상한과 평가 기준을 고정합니다.

In [7]:
# 세 버전을 같은 system 메시지와 모델 설정으로 한 번씩 실행합니다.
runs = {}
for version, prompt_text in versions.items():
    runs[version] = run([("system", "고객 지원 요청을 처리하세요."),
                         ("user", prompt_text)])
    print(version, runs[version]["text"])

v1-direct **분류:** 교환 요청 / 파손 상품 문의

**필요한 다음 행동:**  
파손된 뚜껑 사진을 보내주시면 교환 접수를 도와드리겠습니다. 주문번호 **A-1042** 확인했습니다.

**고객에게 보낼 한 문장:**  
주문번호 A-1042 확인했습니다. 교환 접수를 위해 깨진 뚜껑의 파손 사진을 보내주시면 바로 새 상품 교환 절차를 진행해드리겠습니다.


v2-constraints 분류: 제품 파손 교환 문의

다음 행동: 파손 부위가 보이도록 뚜껑 사진을 주문번호 A-1042와 함께 보내주시면 교환 접수를 도와드리겠습니다.

고객 문장: 어제 받은 텀블러 뚜껑 파손으로 불편을 드려 죄송하며, 주문번호 A-1042와 함께 파손 사진을 보내주시면 확인 후 교환 접수를 진행하겠습니다.


v3-example 분류: 교환  
다음 행동: 파손 사진과 주문번호 A-1042를 확인한 뒤 교환 접수를 진행합니다.  
고객 문장: 파손된 뚜껑 사진을 보내주시면 주문번호 A-1042 기준으로 교환 접수를 도와드리겠습니다.


## 토큰·지연·자동 확인 결과

단어 포함 수 동점도 결과입니다. 한 번의 실행으로 전략 전체의 우열을 확정하지 않습니다.

In [8]:
# usage_metadata의 토큰과 지연, 단어 표시 수를 버전별 행으로 만듭니다.
def estimate_openai_cost(usage):
    input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens")
    if input_tokens is None or output_tokens is None:
        return None
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    normal_input = input_tokens - cache_read
    return (normal_input * 0.75 + cache_read * 0.075 + output_tokens * 4.50) / 1_000_000

table = []
for version, result in runs.items():
    usage = result["usage"] or {}
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    cost = estimate_openai_cost(usage)
    table.append({"version": version, "change": change_log[version],
                  "input_tokens": usage.get("input_tokens"), "output_tokens": usage.get("output_tokens"),
                  "cache_read_tokens": cache_read,
                  "estimated_cost_usd": None if cost is None else round(cost, 8),
                  "latency_ms": round(result["latency_s"] * 1000, 1),
                  "mention_count": sum(mention_flags(result["text"]).values()),
                  "generation_evaluation": "human_review_required"})
print(json.dumps(table, ensure_ascii=False, indent=2))

[
  {
    "version": "v1-direct",
    "change": "기준선",
    "input_tokens": 95,
    "output_tokens": 113,
    "cache_read_tokens": 0,
    "estimated_cost_usd": 0.00057975,
    "latency_ms": 1560.0,
    "mention_count": 3,
    "generation_evaluation": "human_review_required"
  },
  {
    "version": "v2-constraints",
    "change": "형식·금지 조건",
    "input_tokens": 118,
    "output_tokens": 108,
    "cache_read_tokens": 0,
    "estimated_cost_usd": 0.0005745,
    "latency_ms": 1165.0,
    "mention_count": 3,
    "generation_evaluation": "human_review_required"
  },
  {
    "version": "v3-example",
    "change": "v2 조건 유지·관련 예시 추가",
    "input_tokens": 139,
    "output_tokens": 71,
    "cache_read_tokens": 0,
    "estimated_cost_usd": 0.00042375,
    "latency_ms": 870.0,
    "mention_count": 3,
    "generation_evaluation": "human_review_required"
  }
]


## 버전 기록 저장과 재확인

프롬프트 원문, 고정 조건, 원문 응답을 상대 경로의 JSON 파일에 함께 저장합니다.

In [9]:
# 조건별 원문과 측정값을 이름과 함께 출력하여 비교합니다.
from datetime import datetime, timezone

run_at = datetime.now(timezone.utc)
session_id = run_at.strftime("%Y%m%dT%H%M%S%fZ")
record = {"run_id": f"ch07-prompt-ab-{session_id}", "run_date": run_at.date().isoformat(),
          "source_chapter": "ch07", "source_record": {"path": str(ch06_relative), "sha256": ch06_sha256, "run_id": ch06_record["run_id"]},
          "model": "openai:gpt-5.4-mini", "settings": {"max_tokens": 220, "repetitions": 1,
          "pricing_date": ch06_record["pricing"]["date"],
          "pricing_usd_per_million": ch06_record["pricing"]["openai"]},
          "input": TASK, "mention_screen": MENTIONS,
          "comparison_policy": "mention_count는 원문 정독 보조이며 생성 답변 품질 점수가 아님",
          "versions": {name: {"prompt": versions[name], "change": change_log[name],
                               "result": runs[name], "metrics": table[index]}
                       for index, name in enumerate(versions)}}
runs_dir = course_root / "labs/part2/ch07-prompt-engineering/runs"
record_path = runs_dir / "prompt-ab-continuity.json"
session_path = runs_dir / "continuity" / session_id / record_path.name
session_path.parent.mkdir(parents=True, exist_ok=False)
payload = json.dumps(record, ensure_ascii=False, indent=2)
record_path.write_text(payload, encoding="utf-8")
session_path.write_text(payload, encoding="utf-8")
print(record_path.relative_to(course_root), "보존 사본:", session_path.relative_to(course_root))

labs/part2/ch07-prompt-engineering/runs/prompt-ab-continuity.json 보존 사본: labs/part2/ch07-prompt-engineering/runs/continuity/20260909T174009751770Z/prompt-ab-continuity.json
